# DPO (Direct Preference Optimization)

本notebook演示DPO算法的实现和应用。

DPO是一种无需奖励模型的对齐算法，直接从人类偏好数据优化策略。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional

np.random.seed(42)

## 1. 核心函数定义

In [ ]:
def sigmoid(x):
    """Sigmoid函数"""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def log_sigmoid(x):
    """数值稳定的log sigmoid"""
    return -np.log1p(np.exp(-np.clip(x, -500, 500)))

## 2. DPO训练器实现

In [ ]:
class DPOTrainer:
    """DPO训练器"""
    
    def __init__(self, beta: float = 0.1, reference_free: bool = False):
        self.beta = beta
        self.reference_free = reference_free
    
    def dpo_loss(
        self,
        policy_chosen_logps: np.ndarray,
        policy_rejected_logps: np.ndarray,
        reference_chosen_logps: Optional[np.ndarray] = None,
        reference_rejected_logps: Optional[np.ndarray] = None
    ) -> Tuple[float, Dict[str, float]]:
        """计算DPO损失"""
        if self.reference_free:
            logits = self.beta * (policy_chosen_logps - policy_rejected_logps)
        else:
            pi_logratios = policy_chosen_logps - policy_rejected_logps
            ref_logratios = reference_chosen_logps - reference_rejected_logps
            logits = self.beta * (pi_logratios - ref_logratios)
        
        # DPO损失
        losses = -log_sigmoid(logits)
        loss = np.mean(losses)
        
        # 计算隐式奖励
        chosen_rewards = self.beta * (policy_chosen_logps - reference_chosen_logps) \
                         if not self.reference_free else self.beta * policy_chosen_logps
        rejected_rewards = self.beta * (policy_rejected_logps - reference_rejected_logps) \
                          if not self.reference_free else self.beta * policy_rejected_logps
        
        # 收集指标
        metrics = {
            'loss': loss,
            'reward_margin': np.mean(chosen_rewards - rejected_rewards),
            'reward_chosen': np.mean(chosen_rewards),
            'reward_rejected': np.mean(rejected_rewards),
            'accuracy': np.mean((chosen_rewards > rejected_rewards).astype(float)),
            'logits': np.mean(logits),
        }
        
        return loss, metrics
    
    def compute_preference_probability(
        self,
        policy_chosen_logps: float,
        policy_rejected_logps: float,
        reference_chosen_logps: Optional[float] = None,
        reference_rejected_logps: Optional[float] = None
    ) -> float:
        """计算偏好概率 P(y_w > y_l | x)"""
        if self.reference_free:
            logit = self.beta * (policy_chosen_logps - policy_rejected_logps)
        else:
            pi_ratio = policy_chosen_logps - policy_rejected_logps
            ref_ratio = reference_chosen_logps - reference_rejected_logps
            logit = self.beta * (pi_ratio - ref_ratio)
        
        return sigmoid(logit)

## 3. 模拟实验

In [ ]:
# 创建DPO训练器
beta = 0.1
dpo_trainer = DPOTrainer(beta=beta)

print(f"DPO配置:")
print(f"  β (KL惩罚系数): {beta}")
print(f"  作用: 控制策略偏离参考模型的程度")

## 4. 损失计算示例

In [ ]:
# 模拟一个批次的对数概率
batch_size = 4

# 策略模型的对数概率
policy_chosen_logps = np.array([-2.5, -3.0, -2.8, -2.6])
policy_rejected_logps = np.array([-4.5, -5.0, -4.8, -4.6])

# 参考模型的对数概率
reference_chosen_logps = np.array([-3.0, -3.5, -3.2, -3.1])
reference_rejected_logps = np.array([-4.0, -4.5, -4.3, -4.2])

# 计算损失
loss, metrics = dpo_trainer.dpo_loss(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps
)

print(f"DPO损失: {loss:.4f}")
print(f"\n详细指标:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

## 5. 不同β值的影响

In [ ]:
betas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
losses = []
reward_margins = []
accuracies = []

for beta_val in betas:
    trainer = DPOTrainer(beta=beta_val)
    loss_val, metrics_val = trainer.dpo_loss(
        policy_chosen_logps,
        policy_rejected_logps,
        reference_chosen_logps,
        reference_rejected_logps
    )
    losses.append(loss_val)
    reward_margins.append(metrics_val['reward_margin'])
    accuracies.append(metrics_val['accuracy'])

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(betas, losses, 'o-')
axes[0].set_xlabel('β')
axes[0].set_ylabel('Loss')
axes[0].set_title('DPO损失 vs β')
axes[0].grid(True)

axes[1].plot(betas, reward_margins, 'o-', color='green')
axes[1].set_xlabel('β')
axes[1].set_ylabel('Reward Margin')
axes[1].set_title('奖励差距 vs β')
axes[1].grid(True)

axes[2].plot(betas, accuracies, 'o-', color='red')
axes[2].set_xlabel('β')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('准确率 vs β')
axes[2].grid(True)
axes[2].set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

print("观察:")
print("  • β越大，损失越小（更保守的更新）")
print("  • β控制了KL散度的权重")
print("  • 典型值：β ∈ [0.1, 0.5]")

## 6. 偏好概率可视化

In [ ]:
# 计算不同情况下的偏好概率
log_ratio_range = np.linspace(-5, 5, 100)

fig, ax = plt.subplots(figsize=(10, 6))

for beta_val in [0.1, 0.2, 0.5, 1.0]:
    probs = sigmoid(beta_val * log_ratio_range)
    ax.plot(log_ratio_range, probs, label=f'β={beta_val}')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('log(π/π_ref)_chosen - log(π/π_ref)_rejected')
ax.set_ylabel('P(chosen > rejected | x)')
ax.set_title('偏好概率 vs 对数比率（不同β值）')
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

print("解释:")
print("  • x轴: 策略相对于参考模型的对数比率差")
print("  • y轴: 模型预测chosen优于rejected的概率")
print("  • β越大，曲线越陡峭（更自信的偏好）")

## 7. DPO vs RLHF对比

In [ ]:
comparison_data = {
    'Method': ['RLHF (PPO)', 'DPO'],
    'Training Steps': [2, 1],
    'Stability': ['Medium', 'High'],
    'Complexity': ['High', 'Low'],
    'Efficiency': ['Medium', 'High'],
}

print("DPO vs RLHF对比:\n")
print("特性\t\t\tRLHF (PPO)\t\tDPO")
print("=" * 60)
print("训练步骤\t\t2步（RM + RL）\t\t1步")
print("稳定性\t\t\t中等\t\t\t高")
print("复杂度\t\t\t高\t\t\t低")
print("计算效率\t\t中等\t\t\t高")
print("奖励模型\t\t需要\t\t\t不需要")
print("超参数调优\t\t困难\t\t\t简单")
print("\nRLHF: 需要训练奖励模型，然后使用PPO优化")
print("DPO:  直接从偏好数据优化，无需奖励模型")

## 8. Reference-Free DPO

In [ ]:
# 对比标准DPO和Reference-Free DPO
dpo_standard = DPOTrainer(beta=0.1, reference_free=False)
dpo_ref_free = DPOTrainer(beta=0.1, reference_free=True)

loss_standard, metrics_standard = dpo_standard.dpo_loss(
    policy_chosen_logps,
    policy_rejected_logps,
    reference_chosen_logps,
    reference_rejected_logps
)

loss_ref_free, metrics_ref_free = dpo_ref_free.dpo_loss(
    policy_chosen_logps,
    policy_rejected_logps
)

print("标准DPO vs Reference-Free DPO:\n")
print(f"标准DPO损失: {loss_standard:.4f}")
print(f"Reference-Free DPO损失: {loss_ref_free:.4f}")
print(f"\n标准DPO奖励差: {metrics_standard['reward_margin']:.4f}")
print(f"Reference-Free DPO奖励差: {metrics_ref_free['reward_margin']:.4f}")

print("\nReference-Free DPO特点:")
print("  ✓ 不需要参考模型")
print("  ✓ 计算更简单")
print("  ✗ 可能偏离参考模型较多")

## 9. 实际应用示例

In [ ]:
print("DPO的实际应用案例:\n")
print("1. Zephyr-7B")
print("   - 基座: Mistral-7B")
print("   - 数据: UltraFeedback偏好数据")
print("   - 结果: 达到GPT-3.5级别的对话能力\n")

print("2. Tulu 2")
print("   - 基座: Llama 2")
print("   - 方法: SFT + DPO")
print("   - 结果: 在多项任务上超越原始模型\n")

print("3. OpenHermes")
print("   - 基座: Mistral-7B")
print("   - 数据: 混合偏好数据集")
print("   - 结果: 开源对话模型标杆\n")

print("典型训练配置:")
print("  • β: 0.1 ~ 0.5")
print("  • 学习率: 5e-7 ~ 1e-6")
print("  • 训练轮数: 1-3 epoch")
print("  • 批次大小: 32-128")

## 10. 总结

In [ ]:
print("DPO的关键要点:\n")
print("核心优势:")
print("  ✓ 无需奖励模型，简化RLHF pipeline")
print("  ✓ 稳定的监督学习，避免RL不稳定性")
print("  ✓ 计算高效，训练速度快")
print("  ✓ 效果可媲美PPO，甚至更好")
print("  ✓ 广泛应用于开源LLM对齐\n")

print("局限性:")
print("  ✗ 需要高质量的偏好对数据")
print("  ✗ 不如RLHF灵活（只能用偏好对）")
print("  ✗ 理论性质还在研究中\n")

print("最佳实践:")
print("  • 从SFT模型初始化")
print("  • 使用高质量偏好数据")
print("  • 从β=0.1开始调优")
print("  • 监控奖励差距和准确率")
print("  • 避免过拟合（1-3 epoch足够）")